In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import numpy as np
from skimage.color import rgb2lab, lab2rgb
import matplotlib.pyplot as plt
import time
import torchvision.models as models

In [ ]:
class FastFlowerDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.data = []
        image_filenames = [f for f in os.listdir(image_dir)]

        for f in image_filenames:
            img_path = os.path.join(image_dir, f)
            image = Image.open(img_path).convert('RGB')

            if transform:
                image = transform(image)

            image_np = np.array(image)
            image_lab = rgb2lab(image_np).astype(np.float32)

            L = image_lab[:, :, 0:1] / 100.0
            ab = (image_lab[:, :, 1:3] + 128.0) / 255.0

            L_tensor = torch.from_numpy(L.transpose((2, 0, 1)))
            ab_tensor = torch.from_numpy(ab.transpose((2, 0, 1)))

            self.data.append((L_tensor, ab_tensor))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

transform = transforms.Compose([
    transforms.Resize((128, 128))
])

full_dataset = FastFlowerDataset(image_dir='Flowers', transform=transform)

train_size = int(0.99 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
print(f"Dataset complete ! Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

In [ ]:
import torch
import torch.nn as nn

class DoubleConv2D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=kernel_size, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = DoubleConv2D(in_channels, out_channels)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        skip = self.conv(x)
        x = self.pool(skip)
        return skip, x

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        # Après concaténation : out_channels (venant de up) + out_channels (venant de skip) = 2 * out_channels
        self.conv = DoubleConv2D(out_channels * 2, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class Goulet(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = DoubleConv2D(in_channels, out_channels)

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, n_channels: int = 1, n_classes: int = 2, depth: int = 3, base_channels: int = 16):
        super().__init__()
        self.depth = depth
        channels = [base_channels * (2**i) for i in range(depth + 1)]

        self.in_down = Down(n_channels, channels[0])

        self.downs = nn.ModuleList([
            Down(channels[i], channels[i+1]) for i in range(depth)
        ])

        self.goulet = Goulet(channels[-1], channels[-1]*2)

        self.ups = nn.ModuleList([
            Up(channels[i] * 2, channels[i]) for i in reversed(range(1, depth + 1))
        ])

        self.out_up = Up(channels[1], channels[0])
        self.out_conv = nn.Conv2d(channels[0], n_classes, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        skips = []

        # Descente initiale
        skip, x = self.in_down(x)
        skips.append(skip)

        for down in self.downs:
            skip, x = down(x)
            skips.append(skip)

        x = self.goulet(x)

        # Boucle de montée
        for up in self.ups:
            x = up(x, skips.pop())

        # Dernière montée avec le tout premier skip (taille d'image originale)
        x = self.out_up(x, skips.pop())

        # Sortie finale (1x1 conv + Sigmoid)
        return self.sigmoid(self.out_conv(x))

In [ ]:
class ResnetUNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()

        weights = models.ResNet18_Weights.DEFAULT
        resnet = models.resnet18(weights=weights)
        resnet.layer
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.upconv2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.out_conv = nn.Conv2d(32, 2, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        enc1_out = self.enc1(x)
        x = self.pool1(enc1_out)

        enc2_out = self.enc2(x)
        x = self.pool2(enc2_out)

        x = self.bottleneck(x)

        x = self.upconv1(x)
        x = torch.cat([x, enc2_out], dim=1)
        x = self.dec1(x)

        x = self.upconv2(x)
        x = torch.cat([x, enc1_out], dim=1)
        x = self.dec2(x)

        x = self.out_conv(x)
        return self.sigmoid(x)


class ResNetUNet(nn.Module):
    def __init__(self):
        super(ResNetUNet, self).__init__()
        
        # 1. Chargement de ResNet-18 pré-entraîné
        weights = models.ResNet18_Weights.DEFAULT
        resnet = models.resnet18(weights=weights)
        
        # 2. Adaptation de la première convolution (1 canal L au lieu de 3)
        # Moyenne des poids RGB pour conserver l'initialisation
        w = resnet.conv1.weight.data
        new_conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        new_conv1.weight.data = w.sum(dim=1, keepdim=True)
        resnet.conv1 = new_conv1
        
        # 3. Gel des paramètres de l'encodeur
        for param in resnet.parameters():
            param.requires_grad = False
            
        # Découpage de l'encodeur pour récupérer les skip-connections
        self.enc0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu) # (64, 64, 64)
        self.pool = resnet.maxpool                                      # (64, 32, 32)
        self.enc1 = resnet.layer1                                       # (64, 32, 32)
        self.enc2 = resnet.layer2                                       # (128, 16, 16)
        self.enc3 = resnet.layer3                                       # (256, 8, 8)
        self.bottleneck = resnet.layer4                                 # (512, 4, 4)

        # 4. Décodeur (chemins montants)
        # 4 -> 8
        self.upconv4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        
        # 8 -> 16
        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        
        # 16 -> 32
        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        # 32 -> 64
        self.upconv1 = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        
        # 64 -> 128 (résolution d'origine)
        self.upconv0 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.final = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 2, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Descente (Encodeur ResNet)
        e0 = self.enc0(x)          # -> (64, 64, 64)
        p = self.pool(e0)          # -> (64, 32, 32)
        e1 = self.enc1(p)          # -> (64, 32, 32)
        e2 = self.enc2(e1)         # -> (128, 16, 16)
        e3 = self.enc3(e2)         # -> (256, 8, 8)
        b = self.bottleneck(e3)    # -> (512, 4, 4)

        # Montée (Décodeur + Concaténations)
        d4 = self.dec4(torch.cat([self.upconv4(b), e3], dim=1))
        d3 = self.dec3(torch.cat([self.upconv3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.upconv2(d3), e1], dim=1))
        d1 = self.dec1(torch.cat([self.upconv1(d2), e0], dim=1))
        
        out = self.final(self.upconv0(d1))
        return out

In [ ]:
class ColorDiscriminator(nn.Module):
    def __init__(self):
        super(ColorDiscriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 1, kernel_size=4, padding=1)
        )

    def forward(self, L, ab):
        x = torch.cat([L, ab], dim=1)
        return self.model(x)

In [ ]:
#v1 avec mse
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 15
print(device)

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    start_time = time.time()

    for L_batch, ab_batch in train_loader:
        L_batch, ab_batch = L_batch.to(device), ab_batch.to(device)

        optimizer.zero_grad()

        outputs = model(L_batch)

        loss = criterion(outputs, ab_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    elapsed_time = time.time() - start_time
    print(f"Époque [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Temps: {elapsed_time:.2f}s")

torch.save(model.state_dict(), 'modele_unet.pth')

In [ ]:
#Passage à L1 car ça force le modèle à faire de vrai choix plutot que de faire la moyenne
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BasicUNet().to(device)
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
print(device)

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    start_time = time.time()

    for L_batch, ab_batch in train_loader:
        L_batch, ab_batch = L_batch.to(device), ab_batch.to(device)

        optimizer.zero_grad()

        outputs = model(L_batch)

        loss = criterion(outputs, ab_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    elapsed_time = time.time() - start_time
    print(f"Époque [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Temps: {elapsed_time:.2f}s")

torch.save(model.state_dict(), 'modele_unet.pth')

In [ ]:
# GAN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BasicUNet().to(device)
netD = ColorDiscriminator().to(device)

criterion_GAN = nn.BCEWithLogitsLoss()
criterion_L1 = nn.L1Loss()

lambda_L1 = 75.0
lr = 2e-4
betas = (0.5, 0.999)

optimizer_G = optim.Adam(model.parameters(), lr=lr, betas=betas)
optimizer_D = optim.Adam(netD.parameters(), lr=lr, betas=betas)

num_epochs = 80

history_loss_D = []
history_loss_G = []

for epoch in range(num_epochs):
    model.train()
    netD.train()

    start_time = time.time()

    running_loss_G = 0.0
    running_loss_D = 0.0

    for L_batch, ab_batch in train_loader:
        L_batch, ab_batch = L_batch.to(device), ab_batch.to(device)

        fake_ab = model(L_batch)

        optimizer_D.zero_grad()

        pred_real = netD(L_batch, ab_batch)
        loss_D_real = criterion_GAN(pred_real, torch.ones_like(pred_real))

        pred_fake = netD(L_batch, fake_ab.detach())
        loss_D_fake = criterion_GAN(pred_fake, torch.zeros_like(pred_fake))

        loss_D = (loss_D_real + loss_D_fake) * 0.5
        loss_D.backward()
        optimizer_D.step()

        optimizer_G.zero_grad()

        pred_fake_for_G = netD(L_batch, fake_ab)

        loss_G_GAN = criterion_GAN(pred_fake_for_G, torch.ones_like(pred_fake_for_G))

        loss_G_L1 = criterion_L1(fake_ab, ab_batch) * lambda_L1

        loss_G = loss_G_GAN + loss_G_L1
        loss_G.backward()
        optimizer_G.step()

        running_loss_D += loss_D.item()
        running_loss_G += loss_G.item()

    avg_loss_D = running_loss_D / len(train_loader)
    avg_loss_G = running_loss_G / len(train_loader)

    history_loss_D.append(avg_loss_D)
    history_loss_G.append(avg_loss_G)

    elapsed_time = time.time() - start_time

    print(f"Époque [{epoch+1}/{num_epochs}] | Loss D: {running_loss_D/len(train_loader):.4f} | Loss G: {running_loss_G/len(train_loader):.4f}, Temps: {elapsed_time:.2f}s")

fig, ax1 = plt.subplots(figsize=(10, 5))

# Axe gauche (Loss D)
color_D = 'tab:blue'
ax1.set_xlabel('Époque', fontweight='bold')
ax1.set_ylabel('Loss Discriminateur (BCE)', color=color_D, fontweight='bold')
line1 = ax1.plot(range(1, num_epochs + 1), history_loss_D, color=color_D, label='Loss D', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color_D)

# Axe droit (Loss G)
ax2 = ax1.twinx()
color_G = 'tab:orange'
ax2.set_ylabel('Loss Générateur (GAN + L1)', color=color_G, fontweight='bold')
line2 = ax2.plot(range(1, num_epochs + 1), history_loss_G, color=color_G, label='Loss G', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color_G)

# Fusion des légendes
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)

plt.title("Évolution des pertes du GAN", fontsize=14, pad=15)
fig.tight_layout()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
def erreur_chromatique(model, dataloader, device):
    model.eval()
    total_delta = 0.0
    total_pixels = 0

    with torch.no_grad():
        for L_batch, ab_batch in test_loader:
            L_batch, ab_batch = L_batch.to(device), ab_batch.to(device)

            preds = model(L_batch)

            # Dénormalisation des tenseurs (-128 à +127)
            ab_true_unnorm = ab_batch * 255.0 - 128.0
            ab_pred_unnorm = preds * 255.0 - 128.0

            delta = torch.sqrt(torch.sum((ab_true_unnorm - ab_pred_unnorm)**2, dim=1))

            total_delta += torch.sum(delta).item()
            total_pixels += delta.numel()

    mean_delta_ab = total_delta / total_pixels
    print(f"Erreur chromatique moyenne (Delta ab) : {mean_delta_ab:.4f}")

def dispersion_chromatique(model, dataloader, device):
    model.eval()
    ab_values = []

    with torch.no_grad():
        for L_batch, _ in dataloader:  # 'dataloader' au lieu de 'test_loader'
            L_batch = L_batch.to(device)
            preds = model(L_batch)

            # Dénormalisation (-128 à +127)
            ab_pred_unnorm = preds * 255.0 - 128.0
            ab_values.append(ab_pred_unnorm.cpu().numpy())

    ab_values = np.concatenate(ab_values, axis=0)  # Forme : (N, 2, H, W)

    # Réduction sur tous les axes sauf le canal (axe 1)
    dispersion = np.std(ab_values, axis=(0, 2, 3))

    print(
        f"Dispersion chromatique (std dev) : σ_a = {dispersion[0]:.2f}, σ_b = {dispersion[1]:.2f}"
    )
    return dispersion

erreur_chromatique(model, test_loader, device)
dispersion_chromatique(model, test_loader, device)

In [ ]:
def visualize_all_colorization(model, dataloader, device):
    model.eval()

    # 1. On récupère le nombre total d'images dans le dataloader
    total_images = len(dataloader.dataset)
    images_so_far = 0

    print(f"Génération des visualisations pour les {total_images} images du set de test...")

    # 2. On crée dynamiquement la figure avec autant de lignes que d'images
    fig, axes = plt.subplots(total_images, 3, figsize=(12, 4 * total_images))

    # Sécurité au cas où il n'y aurait qu'une seule image dans le set de test
    if total_images == 1:
        axes = [axes]

    with torch.no_grad():
        for L_batch, ab_batch in dataloader:
            L_batch, ab_batch = L_batch.to(device), ab_batch.to(device)
            preds = model(L_batch)

            for i in range(L_batch.size(0)):
                # Récupération des tenseurs individuels sur le CPU
                L = L_batch[i].cpu().numpy().transpose(1, 2, 0)
                ab_true = ab_batch[i].cpu().numpy().transpose(1, 2, 0)
                ab_pred = preds[i].cpu().numpy().transpose(1, 2, 0)

                # Dénormalisation
                L_unnorm = L * 100.0
                ab_true_unnorm = ab_true * 255.0 - 128.0
                ab_pred_unnorm = ab_pred * 255.0 - 128.0

                # Reconstruction des images Lab (H, W, 3)
                lab_true = np.concatenate([L_unnorm, ab_true_unnorm], axis=2)
                lab_pred = np.concatenate([L_unnorm, ab_pred_unnorm], axis=2)

                # Conversion Lab vers RGB
                rgb_true = lab2rgb(lab_true)
                rgb_pred = lab2rgb(lab_pred)

                # Image en nuances de gris pour l'affichage (L dupliqué sur 3 canaux)
                gray_display = np.concatenate([L, L, L], axis=2)

                # Affichage Matplotlib
                ax = axes[images_so_far]

                ax[0].imshow(gray_display)
                ax[0].set_title(f"Entrée N&B (Image {images_so_far+1})")
                ax[0].axis("off")

                ax[1].imshow(rgb_pred)
                ax[1].set_title(f"Prédiction (Image {images_so_far+1})")
                ax[1].axis("off")

                ax[2].imshow(rgb_true)
                ax[2].set_title(f"Image Réelle (Image {images_so_far+1})")
                ax[2].axis("off")

                images_so_far += 1

    plt.tight_layout()
    plt.show()

# Lancement de la visualisation sur tout le test_loader
visualize_all_colorization(model, test_loader, device)